# Can we predict whether a student passes the closing quiz?
Protocol: `protocol.yaml` (group holdout on `student_key`, 5-fold GroupKFold CV on training rows, choose by mean CV ROC AUC, score the holdout once).

In [ ]:
import hashlib, json, yaml
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

proto = yaml.safe_load(open("protocol.yaml"))
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
assert sha("data.csv") == proto["dataset"]["sha256"]
assert sha("holdout.csv") == proto["split"]["holdout_sha256"]
df = pd.read_csv("data.csv")
assert len(df) == proto["dataset"]["rows"]
hold_pos = pd.read_csv("holdout.csv")["row"].to_numpy()   # 0-based positions in data.csv
assert len(hold_pos) == proto["split"]["holdout_rows"]
is_hold = np.zeros(len(df), bool); is_hold[hold_pos] = True
train_pos = np.flatnonzero(~is_hold)
target = proto["target"]
# no student appears on both sides of the split
assert not set(df.student_key[is_hold]) & set(df.student_key[~is_hold])
print(len(train_pos), len(hold_pos))

## Features and prediction time
The prediction is made when the closing quiz starts.

- `enrol_year`, `school_code`, `prior_score` (placement test, taken before the session): known in advance. Kept.
- `hints_used`: counted while the student works through the session, so known when the quiz begins. Kept. (Assumption: the count does not include hints opened during the quiz itself.)
- `session_minutes`: length of the whole session, which ends with the quiz, so it is only final after the outcome. **Excluded.** It is the most correlated column with the target (r = 0.14), which is what you would expect from a post-outcome column.
- `student_key`: excluded by protocol (entity key).

`school_code` is kept numeric.

In [ ]:
features = ["enrol_year", "school_code", "prior_score", "hints_used"]
X, y, groups = df[features], df[target], df["student_key"]

## Candidates (named before any scoring)

In [ ]:
candidates = {
    "baseline_prior": DummyClassifier(strategy="prior"),
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=1),
    "hist_gbm": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
list(candidates)

## Cross-validation on training rows only
GroupKFold by student; scaling is inside the pipeline so it is fitted per fold.

In [ ]:
Xtr, ytr, gtr = X.iloc[train_pos], y.iloc[train_pos], groups.iloc[train_pos]
folds = list(GroupKFold(n_splits=5).split(Xtr, ytr, gtr))
cv_scores = {}
for name, est in candidates.items():
    s = []
    for tr, va in folds:
        m = clone(est).fit(Xtr.iloc[tr], ytr.iloc[tr])
        s.append(float(roc_auc_score(ytr.iloc[va], m.predict_proba(Xtr.iloc[va])[:, 1])))
    cv_scores[name] = s
cv = pd.DataFrame(cv_scores).T
cv["mean"] = cv.mean(axis=1)
cv.round(4)

In [ ]:
chosen = cv["mean"].idxmax()
print("chosen:", chosen, round(cv.loc[chosen, "mean"], 4))

## One holdout evaluation of the chosen candidate (refit on all training rows)

In [ ]:
final = clone(candidates[chosen]).fit(Xtr, ytr)
p = final.predict_proba(X.iloc[hold_pos])[:, 1]
holdout_auc = float(roc_auc_score(y.iloc[hold_pos], p))
print("holdout roc_auc:", round(holdout_auc, 4))

In [ ]:
evidence = {
    "features": features,
    "candidates": list(candidates),
    "cv_folds": [[[int(train_pos[j]) for j in tr], [int(train_pos[j]) for j in va]] for tr, va in folds],
    "cv_scores": cv_scores,
    "chosen": chosen,
    "holdout_rows": [int(i) for i in hold_pos],
    "holdout_scores": {chosen: holdout_auc},
}
json.dump(evidence, open("evidence.json", "w"))